# Bio-Rad CFX Opus

```{device-card} bio-rad-cfx-opus
```

The Bio-Rad CFX Opus is a real-time PCR (qPCR) system. This driver controls it directly over its
USB Type-B port, with no Bio-Rad software (no CFX Maestro, no BR.io) on the controlling computer.
It authors a thermal protocol, starts a run, and collects each plate read's fluorescence as the
run proceeds.

```{warning}
This driver has not yet been verified against a real CFX Opus. Its command language and USB
framing are carried over from the related CFX96 / C1000 Touch base; the Opus's USB product ID and
exact reply formats may differ. `setup()` logs a warning until a run confirms it. Please report a
successful or failed run on the [PyLabRobot forum](https://discuss.pylabrobot.org).
```

## How it talks

The instrument speaks an ASCII command language over USB bulk endpoints: a protocol is typed in
one directive at a time, started with a single run command, and each plate read is written to the
instrument's storage as a file the host downloads. A once-a-second status poll reports the block
and lid temperatures, the current step, and the run's progress. You drive all of this through the
`CFXOpus` class and never touch the wire format yourself.

## Physical setup

1. Connect the CFX Opus to the computer with the supplied USB Type-B cable (rear panel).
2. Power the instrument on and wait for its touchscreen to finish starting.
3. Make sure no Bio-Rad software is connected to the instrument at the same time: it accepts one
   remote client at a time.

On Linux you may need a udev rule granting access to Bio-Rad's USB vendor ID (`0x0614`); on
Windows, PyLabRobot uses the bundled libusb backend.

## Finding the instrument

If you do not know the Opus's USB product ID, list the connected Bio-Rad devices from a terminal:

```
plr dev cfx-opus list
```

Pass the product ID you see to `CFXOpus(id_product=...)`. The default is the CFX96/C1000 family ID.

## Connecting

`setup()` opens the USB connection and identifies the instrument.

In [ ]:
from pylabrobot.biorad import CFXOpus

opus = CFXOpus()
await opus.setup()

print(await opus.identify())

## Reading status

Read the block's temperatures, current step and run state at any time. `get_optics_status()`
reports the optical head.

In [ ]:
status = await opus.get_status()
print(status.block_temperature, status.lid_temperature, status.run_state)

print(await opus.get_optics_status())

## Opening and closing the lid

Lid control is idempotent: `open_lid()` and `close_lid()` read the current position and act only
if needed, so they are safe to call repeatedly.

In [ ]:
await opus.open_lid()

Load your sealed plate into the block, then close the lid.

In [ ]:
await opus.close_lid()

## Running a protocol

Build a protocol from steps. A `Cycle` repeats its steps (here 40 cycles of denature, anneal and
read) and a `PlateRead` measures fluorescence. `volume` is the per-well sample volume in µL; with a
volume the instrument controls the calculated sample temperature.

`run_protocol` authors the protocol, starts the run, waits for it to finish while collecting each
plate read, and returns the reads in order. The heated lid warms up before the block moves, so the
first minutes show no block temperature change — this is normal.

In [ ]:
from pylabrobot.biorad import Cycle, Hold, PlateRead, Protocol

protocol = Protocol(
    steps=[
        Hold(95, 180),
        Cycle([Hold(95, 10), Hold(60, 30), PlateRead()], count=40),
    ],
    volume=20,
)

reads = await opus.run_protocol(protocol, run_name="demo")
print(f"collected {len(reads)} plate reads")

Each read carries the decoded fluorescence. `read.mean(channel)` is a row-major grid of mean
fluorescence (well A1 first), and `read.well(channel, "A1")` reads one well.

In [ ]:
last = reads[-1].read
print("cycle", last.cycle, "channels", last.channels)
print("A1 on channel 1:", last.well(1, "A1").mean)

## Melt curves and gradients

A `MeltCurve` raises the temperature across a range, reading at every step. A `Gradient` holds a
temperature gradient across the block's rows — useful for optimizing an annealing temperature.

In [ ]:
from pylabrobot.biorad import Gradient, MeltCurve

optimisation = Protocol(
    steps=[
        Hold(95, 180),
        Cycle([Hold(95, 10), Gradient(55, 65, 30), PlateRead()], count=35),
        MeltCurve(start=65, end=95, increment=0.5),
    ],
    volume=20,
)

## Stopping early

To stop a run in progress, call `stop_run()`. During a run you can also `pause_run()` and
`resume_run()`.

In [ ]:
# await opus.stop_run()

## Disconnecting

In [ ]:
await opus.stop()